In [1]:
import pandas as pd
import numpy as np

# 1. Load Relational Tables
print("Ingesting Olist Datasets...")
orders = pd.read_csv('olist_orders_dataset.csv')
items = pd.read_csv('olist_order_items_dataset.csv')
geo = pd.read_csv('olist_geolocation_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')

# 2. Geospatial Cleanup (Average coordinates per zip code to remove outliers)
geo_clean = geo.groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']].mean().reset_index()

# 3. Master Merge Pipeline
df = orders.merge(items, on='order_id', how='inner')
df = df.merge(customers[['customer_id', 'customer_zip_code_prefix']], on='customer_id', how='inner')
df = df.merge(geo_clean, left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left')

# Rename for spatial clarity
df.rename(columns={'geolocation_lat': 'customer_lat', 'geolocation_lng': 'customer_lng'}, inplace=True)

# 4. Temporal Engineering (Convert to Datetime)
time_cols = ['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in time_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# --- ELITE DATA CLEANING STEPS ---
# Remove canceled or unavailable orders
df = df[df['order_status'] == 'delivered']
# Remove "Time Travel" errors
df = df[df['order_delivered_customer_date'] > df['order_purchase_timestamp']]
# Remove geographic outliers (coordinates outside Brazil's bounding box)
df = df[(df['customer_lat'] <= 5.27) & (df['customer_lat'] >= -33.75)]
df = df[(df['customer_lng'] <= -34.80) & (df['customer_lng'] >= -73.98)]

# 5. Bottleneck & Financial Loss Calculation
# Calculate exact delay in days (Actual Delivery - Estimated Delivery)
df['delivery_delta_days'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.total_seconds() / (24 * 3600)

# Flag delayed orders (Delta > 0 means it arrived late)
df['is_delayed'] = df['delivery_delta_days'] > 0

# Calculate Financial Capital Tied up in Delays (Freight Cost + Product Value)
df['financial_loss_exposure'] = np.where(df['is_delayed'], (df['price'] + df['freight_value']) * 0.15, 0)

# 6. Filter to relevant active deliveries
logistics_master = df.dropna(subset=['customer_lat', 'customer_lng', 'order_delivered_customer_date']).copy()

total_loss = logistics_master['financial_loss_exposure'].sum()
delay_rate = (logistics_master['is_delayed'].sum() / len(logistics_master)) * 100

print(f"Pipeline Complete. Total Records: {len(logistics_master)}")
print(f"System-Wide Delay Rate: {delay_rate:.2f}%")
print(f"Estimated Capital Lost to Inefficiencies: ${total_loss:,.2f}")

Ingesting Olist Datasets...
Pipeline Complete. Total Records: 109891
System-Wide Delay Rate: 7.90%
Estimated Capital Lost to Inefficiencies: $202,155.10


In [ ]:
# --- Cell 2: Spatial Mapping of Logistics Bottlenecks ---
import folium
from folium.plugins import HeatMap

print("Mapping severe financial bottlenecks...")

# 1. Isolate the 7.9% of delayed deliveries
delayed_deliveries = logistics_master[logistics_master['is_delayed'] == True].copy()

# 2. Aggregate the financial loss by exact geographic coordinates
bottlenecks = delayed_deliveries.groupby(['customer_lat', 'customer_lng']).agg(
    total_delays=('is_delayed', 'count'),
    total_financial_loss=('financial_loss_exposure', 'sum')
).reset_index()

# 3. Filter for chronic bottlenecks (locations with 3+ delays) to eliminate noise
severe_bottlenecks = bottlenecks[bottlenecks['total_delays'] >= 3]
print(f"Isolated {len(severe_bottlenecks)} chronic bottleneck zones.")

# 4. Initialize an interactive spatial map centered on Brazil (Updated Tiles)
map_center = [-14.2350, -51.9253] 
delay_map = folium.Map(location=map_center, zoom_start=4, tiles='OpenStreetMap')

# 5. Generate a HeatMap layer weighted by financial loss (darker red = higher capital loss)
heat_data = [[row['customer_lat'], row['customer_lng'], row['total_financial_loss']] 
             for index, row in severe_bottlenecks.iterrows()]

HeatMap(heat_data, radius=12, blur=15, max_zoom=10).add_to(delay_map)

# Save the map as an HTML file for your GitHub portfolio
delay_map.save('logistics_bottleneck_heatmap.html')
print("Saved interactive map as 'logistics_bottleneck_heatmap.html'.")

# Display the map directly in the notebook
delay_map

In [4]:
# --- Cell 3: Spatial Machine Learning (Facility Location Optimization) ---
from sklearn.cluster import KMeans
import pandas as pd

print("Deploying Spatial Clustering to calculate optimal hub locations...")

# 1. Extract the coordinates of our chronic bottlenecks
X = severe_bottlenecks[['customer_lat', 'customer_lng']]

# 2. Initialize K-Means to propose 3 targeted regional distribution hubs
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)

# Suppress the SettingWithCopyWarning by using .loc if needed, but direct assignment is fine for this portfolio script
severe_bottlenecks = severe_bottlenecks.copy() 
severe_bottlenecks['hub_assignment'] = kmeans.fit_predict(X)

# 3. Extract the exact latitude and longitude for the new proposed hubs
proposed_hubs = pd.DataFrame(kmeans.cluster_centers_, columns=['hub_lat', 'hub_lng'])
proposed_hubs.index.name = 'Hub ID'

print("\n--- EXECUTIVE RECOMMENDATION ---")
print("To neutralize localized revenue bleed, acquire distribution centers at these coordinates:")
print(proposed_hubs)

Deploying Spatial Clustering to calculate optimal hub locations...

--- EXECUTIVE RECOMMENDATION ---
To neutralize localized revenue bleed, acquire distribution centers at these coordinates:
          hub_lat    hub_lng
Hub ID                      
0       -8.478798 -40.165781
1      -22.145994 -44.851180
2      -27.256275 -51.030333
